In [0]:
# Section 1 — Read Bronze Customer Table

df = spark.table("bike_lakehouse.bronze.crm_cust_info")

display(df)

In [0]:
df.printSchema()

In [0]:
# Section 1 — Check Row Count

print("Row count:", df.count())

In [0]:
# Section 2 — Data Quality Analysis
# Check for duplicate customer IDs

duplicate_ids = (
    df.groupBy("cst_id")
      .count()
      .filter("count > 1")
)

display(duplicate_ids)

In [0]:
# Section 2 — Check NULL Values

from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
# Section 2 — Inspect Categorical Values

display(
    df.select(
        "cst_marital_status",
        "cst_gndr"
    ).distinct()
)

In [0]:
# Section 2 — Check Duplicate Customer Business Keys

duplicate_keys = (
    df.groupBy("cst_key")
      .count()
      .filter("count > 1")
)

display(duplicate_keys)

In [0]:
# Section 2 — Inspect Duplicate Customer Records

duplicate_keys = [
    "AW00029433",
    "AW00029449",
    "AW00029466",
    "AW00029473",
    "AW00029483"
]

display(
    df.filter(col("cst_key").isin(duplicate_keys))
      .orderBy("cst_key", "cst_id")
)

In [0]:
# Section 2 — Inspect Customer Creation Dates

display(
    df.select("cst_create_date")
      .distinct()
      .orderBy("cst_create_date")
)

In [0]:
# Section 2 — Inspect Missing/Invalid Date Records

display(
    df.filter(col("cst_create_date").isNull())
)

In [0]:
# Section 2 — Inspect Customer ID Range

display(
    df.select("cst_id")
      .summary()
)

In [0]:
# Section 2 — Check Name Whitespace

from pyspark.sql.functions import trim

whitespace_issues = df.filter(
    (col("cst_firstname") != trim(col("cst_firstname"))) |
    (col("cst_lastname") != trim(col("cst_lastname")))
)

display(whitespace_issues)

In [0]:
# Section 2 — Inspect Name Casing

display(
    df.select("cst_firstname", "cst_lastname")
      .filter(
          col("cst_firstname").isNotNull() |
          col("cst_lastname").isNotNull()
      )
      .limit(100)
)

In [0]:
# Section 2 — Inspect Customer Key Format

display(
    df.select("cst_key")
      .filter(col("cst_key").isNotNull())
      .limit(100)
)

In [0]:
# Section 3 — Transformation 1: Trim Customer Names

from pyspark.sql.functions import trim, col

df_clean = (
    df
    .withColumn("cst_firstname", trim(col("cst_firstname")))
    .withColumn("cst_lastname", trim(col("cst_lastname")))
)

display(df_clean.limit(20))

In [0]:
# Sanity Check — Name Whitespace

display(
    df_clean.filter(
        (col("cst_firstname") != trim(col("cst_firstname"))) |
        (col("cst_lastname") != trim(col("cst_lastname")))
    )
)

In [0]:
# Section 3 — Transformation 2: Remove Invalid Customer Records

df_clean = df_clean.filter(
    col("cst_id").isNotNull()
)

display(df_clean)

In [0]:
# Sanity Check — NULL Customer IDs

print("Rows after removing invalid records:", df_clean.count())

display(
    df_clean.filter(col("cst_id").isNull())
)

In [0]:
# Section 3 — Calculate Customer Record Completeness

from pyspark.sql.functions import col, when

df_clean = df_clean.withColumn(
    "completeness_score",
    when(col("cst_id").isNotNull(), 1).otherwise(0)
    + when(col("cst_key").isNotNull(), 1).otherwise(0)
    + when(col("cst_firstname").isNotNull(), 1).otherwise(0)
    + when(col("cst_lastname").isNotNull(), 1).otherwise(0)
    + when(col("cst_marital_status").isNotNull(), 1).otherwise(0)
    + when(col("cst_gndr").isNotNull(), 1).otherwise(0)
    + when(col("cst_create_date").isNotNull(), 1).otherwise(0)
)

display(
    df_clean
    .filter(
        col("cst_key").isin([
            "AW00029433",
            "AW00029449",
            "AW00029466",
            "AW00029473",
            "AW00029483"
        ])
    )
    .orderBy("cst_key", "completeness_score", "cst_create_date")
)

In [0]:
# Section 3 — Transformation 3: Deduplicate Customers

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("cst_key").orderBy(
    desc("completeness_score"),
    desc("cst_create_date")
)

df_clean = (
    df_clean
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num", "completeness_score")
)

display(
    df_clean.filter(
        col("cst_key").isin([
            "AW00029433",
            "AW00029449",
            "AW00029466",
            "AW00029473",
            "AW00029483"
        ])
    )
)

In [0]:
# Sanity Check — Duplicate Customer Keys

duplicate_keys_after = (
    df_clean
    .groupBy("cst_key")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_keys_after)

In [0]:
# Section 3 — Inspect Missing Gender Values

display(
    df_clean
    .filter(col("cst_gndr").isNull())
    .limit(50)
)

In [0]:
# Section 3 — Transformation 4: Standardize Missing Gender

from pyspark.sql.functions import coalesce, lit

df_clean = df_clean.withColumn(
    "cst_gndr",
    coalesce(col("cst_gndr"), lit("Unknown"))
)

display(
    df_clean.select("cst_id", "cst_key", "cst_gndr").limit(50)
)

In [0]:
# Sanity Check — NULL Gender Values

null_gender_count = (
    df_clean
    .filter(col("cst_gndr").isNull())
    .count()
)

print("NULL gender values:", null_gender_count)

In [0]:
# Section 3 — Inspect Missing Marital Status

display(
    df_clean
    .filter(col("cst_marital_status").isNull())
)

In [0]:
# Sanity Check — Customer Creation Date

null_date_count = (
    df_clean
    .filter(col("cst_create_date").isNull())
    .count()
)

print("NULL creation dates:", null_date_count)

In [0]:
# Section 4 — Final Silver DataFrame Check

print("Final row count:", df_clean.count())

df_clean.printSchema()

display(df_clean.limit(20))

In [0]:
# Section 4 — Rename Columns for Silver

df_silver = (
    df_clean
    .withColumnRenamed("cst_id", "customer_id")
    .withColumnRenamed("cst_key", "customer_key")
    .withColumnRenamed("cst_firstname", "first_name")
    .withColumnRenamed("cst_lastname", "last_name")
    .withColumnRenamed("cst_marital_status", "marital_status")
    .withColumnRenamed("cst_gndr", "gender")
    .withColumnRenamed("cst_create_date", "create_date")
)

display(df_silver.limit(10))

In [0]:
# Section 4 — Final Silver Sanity Checks

print("Final row count:", df_silver.count())

print(
    "Duplicate customer keys:",
    df_silver.groupBy("customer_key")
             .count()
             .filter(col("count") > 1)
             .count()
)

print(
    "NULL customer IDs:",
    df_silver.filter(col("customer_id").isNull()).count()
)

print(
    "NULL customer keys:",
    df_silver.filter(col("customer_key").isNull()).count()
)

print(
    "NULL creation dates:",
    df_silver.filter(col("create_date").isNull()).count()
)

print(
    "NULL genders:",
    df_silver.filter(col("gender").isNull()).count()
)

In [0]:
# Section 5 — Write Silver Customer Table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.silver.crm_customers")

In [0]:
# Section 5 — Verify Silver Customer Table

df_verify = spark.table("bike_lakehouse.silver.crm_customers")

print("Silver row count:", df_verify.count())

df_verify.printSchema()

display(df_verify.limit(10))